# Issue 19 — Qyroven independent Source/Development smoke
Training runs on Colab. No Final use, online controller or superiority claim.
Upload an identified Git bundle of `feat/19-qyroven-model-retention` (create with `git bundle create issue19-code.bundle feat/19-qyroven-model-retention`). The bundle contains source history only, no data/credentials. The executed SHA is recorded. Prefer T4 runtime; do not silently substitute local training.


In [ ]:
import hashlib
import json
import os
import pathlib
import subprocess
import sys
import zipfile

from google.colab import files

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "torch==2.11.0", "numpy==2.3.3", "gdown==5.2.0"],
    check=True,
)

uploaded = files.upload()  # select the Git bundle, no access token in notebook cells
assert list(uploaded) == ["issue19-code.bundle"]
print("CODE BUNDLE SHA256", hashlib.sha256(uploaded["issue19-code.bundle"]).hexdigest())
subprocess.run(
    [
        "git",
        "clone",
        "-b",
        "feat/19-qyroven-model-retention",
        "issue19-code.bundle",
        "/content/ev-issue19",
    ],
    check=True,
)
REPO = pathlib.Path("/content/ev-issue19")
print(
    "EXECUTED CODE SHA",
    subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True),
)

In [ ]:
import gdown


def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for b in iter(lambda: stream.read(1048576), b""):
            h.update(b)
    return h.hexdigest()


catalog = json.loads((REPO / "data/releases/DATA-v1-20261019.json").read_text())
package = next(p for p in catalog["packages"] if p["name"] == "inputs-urbanev-SZH-v1.zip")
archive = pathlib.Path("/content") / package["name"]
if not archive.exists():
    gdown.download(id=package["drive_file_id"], output=str(archive), quiet=False)
assert archive.stat().st_size == package["size_bytes"] and sha(archive) == package["sha256"]
DATA = pathlib.Path("/content/ev-data")
if not DATA.exists():
    with zipfile.ZipFile(archive) as z:
        assert all(
            not p.startswith("/") and ".." not in pathlib.PurePosixPath(p).parts
            for p in z.namelist()
        )
        z.extractall(DATA)
print("INPUT ARCHIVE VERIFIED", sha(archive))

## Execute the fixed matrix
Source: all 65,888 eligible UrbanEV Source examples, one epoch, seed 42. Targets: IDs 1001 and 1008 (Development), budgets 7/14/28, fixed lambda 0 and 0.01, three adaptation epochs. Last planned checkpoints only; no best-score selection. Initial epsilon 0.001, MLP width 64, Adam lr 0.001 are declared smoke settings, not tuned official constants. Score the first seven scheduled days without online updates. This does not satisfy #21 replay or Sprint 3.


In [ ]:
import datetime

import torch

assert torch.cuda.is_available(), "Allocate a Colab GPU; do not train on the local machine"
print(sys.version, torch.__version__, torch.cuda.get_device_name(0))
RUN_ID = "qyroven-i19-" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT = pathlib.Path("/content") / RUN_ID
subprocess.run(
    [
        sys.executable,
        "-m",
        "tools.issue19_colab_smoke",
        "--inputs-root",
        str(DATA / "data/inputs/issue18-input-handoff-v1-20261019"),
        "--output",
        str(OUT),
    ],
    cwd=REPO,
    check=True,
)
print("COMPLETE", OUT)

## Persist artifacts before disconnecting
Use a NEW Drive directory per run. Mount only your intended Drive account when authorizing Colab access. Store both final and epoch checkpoints, all logs/configs/metrics/manifests, the input package hash, pip freeze, and SHA256SUMS. The same code/config/device can resume from an epoch checkpoint via `--resume` and `--resume-sha256`; point `--output` to a fresh run directory. Exact cross-device/PyTorch reproducibility is not promised.


In [ ]:
import shutil

from google.colab import drive

subprocess.run(
    [sys.executable, "-m", "pip", "freeze"], stdout=(OUT / "pip-freeze.txt").open("x"), check=True
)
(OUT / "SHA256SUMS.txt").write_text(
    "".join(f"{sha(p)}  {p.relative_to(OUT)}\n" for p in sorted(OUT.rglob("*")) if p.is_file())
)
# Explicit Drive mount requires your normal Colab authorization. No credential is saved here.

drive.mount("/content/drive")

DEST = pathlib.Path("/content/drive/MyDrive/AdaptiveRetentionEV") / RUN_ID
assert not DEST.exists()
shutil.copytree(OUT, DEST)
print("DRIVE ARTIFACT DIRECTORY", DEST)